In [1]:
!apt-get update -qq
!apt-get install -y espeak-ng ffmpeg -qq
!pip install -q sentence-transformers faiss-cpu openai-whisper

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 122363 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.1-6build2_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.1-6build2) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-11build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-11build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.50+dfsg-10ubuntu0.1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.50+dfsg-10ubuntu0.1) ...
Selecting previ

In [2]:
import faiss
import numpy as np
import subprocess
from sentence_transformers import SentenceTransformer
from IPython.display import Audio

# =========================
# 1. Base de conocimiento ampliada
#    Fuente: Código de Trabajo de Guatemala (Decreto 1441)
# =========================
casos = [
    {
        "titulo": "Despido injustificado",
        "texto_es": "Si un trabajador es despedido sin causa justificada (Art. 78 CT), tiene derecho a indemnización equivalente a un mes de salario por cada año de servicio continuo, más el pago de vacaciones, aguinaldo y bono 14 proporcionales. La denuncia puede presentarse en la Inspección General de Trabajo (MINTRAB).",
        "texto_kiche": "We jun ajchak xel b'anik maj k'utunem (Art. 78 CT), k'o ri rajawarem richin k'amowik pwaq: jun ik' pwaq pa jujun junab' xchak pa ri chak. Xuquje' k'o chi tojik ri ejqanem, aguinaldo xuquje' bono 14. Kojkäj pa Inspección General de Trabajo."
    },
    {
        "titulo": "Horas extras",
        "texto_es": "Las horas trabajadas fuera de la jornada ordinaria son horas extraordinarias y deben pagarse con un recargo mínimo del 50% sobre el valor de la hora ordinaria (Art. 121 CT). La jornada diurna y nocturna juntas no pueden exceder 12 horas diarias. El patrono debe registrar las horas extra en sus libros de salarios.",
        "texto_kiche": "Ri taq hora k'utun pa ruwi' ri q'ijil chak rajawaxik rutojik ruk' nik'aj chik ri pwaq pa jun hora (Art. 121 CT). Man k'o chi nuk'is ta pa lajuj kab' (12) hora pa jun q'ij ri chakïk pa q'ij xuquje' pa aq'ab'. Ri ajpatron k'o chi rutz'ib'axik ri taq hora k'utun."
    },
    {
        "titulo": "Vacaciones",
        "texto_es": "Todo trabajador tiene derecho a 15 días hábiles de vacaciones pagadas por cada año de trabajo continuo (Art. 130 CT). Las vacaciones deben tomarse dentro de los 60 días siguientes al año cumplido. Si el trabajador cesa antes del año, recibe compensación proporcional. No pueden compensarse en dinero mientras la relación laboral esté activa.",
        "texto_kiche": "Ronojel ajchak k'o rajawarem richin chib'ojlajuj (15) q'ij e okisaxik chi ejqanem tojonik pa junam jun juna' e chakïk (Art. 130 CT). K'o chi k'amowik ri ejqanem pa wuqajinik (60) q'ij chik rij ri juna'. We kel pa ri chak nab'e juna', k'o rajawarem richin nik'aj ejqanem."
    },
    {
        "titulo": "Aguinaldo",
        "texto_es": "El aguinaldo es obligatorio y equivale a un salario ordinario mensual íntegro (Decreto 76-78). Debe pagarse del 1 al 15 de diciembre de cada año. Si el trabajador laboró menos de un año, le corresponde la parte proporcional al tiempo trabajado.",
        "texto_kiche": "Ri aguinaldo rajawaxik tojik. Jun mes tz'aq'at uwetamab'al (Decreto 76-78). Rajawaxik nub'an chi tojik pa ri q'ij nik'aj lajuj Diciembre pa jujun juna'. We mayub' juna' xab'an chakïk, rajawaxik nik'aj aguinaldo."
    },
    {
        "titulo": "Salario mínimo",
        "texto_es": "El empleador no puede pagar menos del salario mínimo vigente (Art. 103 CT). El salario mínimo se fija anualmente por el Organismo Ejecutivo a través del Ministerio de Trabajo. Para 2025: actividades no agrícolas Q11.74/hora y agrícolas Q11.52/hora. El incumplimiento es sancionable por el MINTRAB.",
        "texto_kiche": "Ri ajpatron man utz taj nutoj jub'a' pwaq chuwäch ri pwaq ri k'o chi tojik pa ruwi' ri ley (Art. 103 CT). Ri tz'aq'at rajawaxik chakïk k'iyin pa ronojel junab' pa ruwi' ri Ministerio de Trabajo. Ri ajpatron ri man nutoj ta ri tz'aq'at uwetamab'al, k'o chi nuk'is ri multa."
    },
    {
        "titulo": "Licencia por maternidad",
        "texto_es": "La trabajadora embarazada tiene derecho a 84 días de descanso remunerado: 30 días antes del parto y 54 días después (Art. 152 CT). El patrono no puede despedirla durante el embarazo ni dentro de los 10 meses posteriores al parto. El padre tiene derecho a 2 días de licencia pagada por el nacimiento del hijo.",
        "texto_kiche": "Ri ixoq ajchak iyom xatilitaj pa wuqub' winaq (84 q'ij) e oq'omaxik chi iyomaxik: wuqajinik (30 q'ij) nab'e ri alaxik, xuquje' lajuj wuqajinik (54 q'ij) chik rij (Art. 152 CT). Man xtz'apij ta ri ixoq iyom pa ri chak. Ri achi ajchak k'o rajawarem richin kab' (2) q'ij richin ri alaxik ri ral."
    },
    {
        "titulo": "Contratos de trabajo",
        "texto_es": "El contrato de trabajo puede ser verbal o escrito (Art. 26 CT). El contrato escrito ofrece mayor protección. Debe incluir: nombre de las partes, descripción del puesto, salario pactado, lugar y duración. En Guatemala existe presunción de relación laboral cuando hay subordinación y dependencia económica. El período de prueba es de 2 meses.",
        "texto_kiche": "Ri ajq'a'nchi' taq tzij yatob'ën pa tzib'b'äl o pa tzij re chak (Art. 26 CT). Ri pa tzib'b'äl k'o ri nima'q rajawaxik. K'o chi nuk'ulb'esaj: ri b'i'aj ri ajchakib', ri tz'aq'at uwetamab'al, ri qas taq tzij. Ri k'atz'inem tz'aq'at uwetamab'al kab' ik' (2 ik')."
    },
    {
        "titulo": "Renuncia voluntaria",
        "texto_es": "Si el trabajador renuncia voluntariamente, no tiene derecho a indemnización. Sin embargo, el patrono debe pagar las prestaciones pendientes: vacaciones no gozadas, proporciones de aguinaldo y bono 14, y cualquier salario adeudado. El trabajador puede también retirarse justificadamente con derecho a indemnización si el patrono incumple el contrato (Art. 79 CT).",
        "texto_kiche": "We ri ajchak kel ruk' ri chak pa ruma ri utzil, man k'o ta indemnización. Pero k'o chi tojik ri rajilab'al k'iyojil k'o chik: ejqanem man xb'an ta, nik'aj aguinaldo xuquje' bono 14, xuquje' ri pwaq k'o chik. We ri ajpatron man nuk'umaj ta ri chak, ri ajchak k'o rajawarem richin indemnización (Art. 79 CT)."
    },
    {
        "titulo": "Jornada laboral",
        "texto_es": "La jornada ordinaria diurna es de 8 horas diarias y 44 horas semanales (Art. 116 CT). La jornada nocturna es de 6 horas diarias y 36 semanales. La jornada mixta es de 7 horas diarias y 42 semanales. La jornada diurna es de 6am a 18hrs, nocturna de 18hrs a 6am. No pueden exceder 12 horas diarias en total incluyendo extraordinarias.",
        "texto_kiche": "Ri chakïk pa q'ij: waqxaqib' (8) hora pa jun q'ij, kuk' kawinaq kab' (44) hora pa junab'aq'a (Art. 116 CT). Ri chakïk pa aq'ab': waqib' (6) hora pa jun q'ij, kuk' wuqajinik waqib' (36) hora pa junab'aq'a. Man k'o chi nuk'is ta pa lajuj kab' (12) hora pa jun q'ij."
    },
    {
        "titulo": "Seguro social IGSS",
        "texto_es": "El patrono debe afiliar a sus trabajadores al Instituto Guatemalteco de Seguridad Social (IGSS) y realizar los aportes correspondientes (Art. 1 Ley IGSS). El trabajador también aporta un porcentaje de su salario. El IGSS cubre: enfermedad, maternidad, invalidez, vejez y accidentes de trabajo. El incumplimiento patronal es sancionable.",
        "texto_kiche": "Ri ajpatron k'o chi rutz'aqatisaxik ri ajchak pa IGSS xuquje' k'o chi rutojik ri k'iyojil (Art. 1 Ley IGSS). Ri ajchak xuquje' k'o nutoj jun b'eyomal pa ri pwaq. Ri IGSS nuchap: yawalib'al, iyomaxik, pixab'anel, xuquje' tzaq'axik pa ri chak."
    },
    {
        "titulo": "Bono 14",
        "texto_es": "El Bono 14 equivale a un salario mensual ordinario y debe pagarse en la primera quincena de julio de cada año (Decreto 42-92). Si el trabajador laboró menos de un año, le corresponde la parte proporcional. No puede sustituirse ni deducirse de otros pagos. Se calcula con base en el salario promedio de los últimos 12 meses.",
        "texto_kiche": "Ri Bono 14 rajawaxik jujun mes tz'aq'at uwetamab'al, nub'an pa ri nik'aj b'elejeb' winaq Julio pa jujun juna' (Decreto 42-92). We mayub' juna' xab'an chakïk, rajawaxik nik'aj bono. Man nitz'aqatex ta rech jun chik pwaq. Nub'an pa ruwi' ri pwaq promedio ri lajuj kab' ik' (12 ik') chik rij."
    },
    {
        "titulo": "Discriminación laboral",
        "texto_es": "Se prohíbe la discriminación por motivo de raza, religión, credos políticos, sexo, situación económica, etnia o cualquier otra índole (Art. 14 BIS y 137 BIS CT). El patrono que discrimine en la contratación o durante la relación laboral puede ser sancionado. La discriminación por embarazo está especialmente protegida.",
        "texto_kiche": "Man utz ta ri k'exb'al ucholaj ajchak pa ruwi' ri amaq', ri keroj, ri tzij politiko, ri ixoq o achi, ri tz'aq'at uwetamab'al o ri amaq' (Art. 14 BIS CT). Ri ajpatron ri nuk'ex cholaj k'o chi nuk'is multa. Ri k'exb'al ucholaj pa ruwi' ri iyomaxik xti'aj nik'aj rajawaxik."
    },
    {
        "titulo": "Días de asueto",
        "texto_es": "Los trabajadores particulares tienen derecho a días de asueto con goce de salario (Art. 127 CT): 1 de enero, Semana Santa (jueves, viernes y sábado), 1 de mayo, 30 de junio, 15 de septiembre, 20 de octubre, 1 de noviembre, 24 de diciembre (medio día), 25 de diciembre, 31 de diciembre (medio día) y día de la festividad local.",
        "texto_kiche": "Ri taq ajchak particulares k'o rajawarem richin taq q'ij asueto ruk' tojib'al pwaq (Art. 127 CT): ri nab'e q'ij Enero, Semana Santa, ri nab'e q'ij Mayo, ri wuqajinik q'ij Junio, ri lajuj wuqub' q'ij Septiembre, ri winaq q'ij Octubre, ri nab'e q'ij Noviembre, ri lajuj kab' q'ij Diciembre xuquje' ri winaq waqlajuj q'ij Diciembre."
    },
    {
        "titulo": "Trabajo de menores de edad",
        "texto_es": "Se prohíbe el trabajo de menores de 14 años (Art. 148 CT). Para los mayores de 14 años la jornada se reduce en 1 hora diaria. Se prohíbe el trabajo nocturno y extraordinario para menores. El MINTRAB puede autorizar excepciones calificadas. Los menores de edad deben cumplir con la obligatoriedad escolar.",
        "texto_kiche": "Man k'o ta chi chak ri achike ajchak man k'ulmataj ta lajuj kaib' (14) junab' (Art. 148 CT). Ri más kab'lajuj (14) junab', ri q'ijil chak nuk'is jun hora pa jun q'ij. Man k'o ta chi chak pa aq'ab' ni jun chak extra ri man k'ulmataj ta lajuj kaib' junab'. K'o chi nab'an chak pa ri tijob'al."
    },
    {
        "titulo": "Suspensión del contrato",
        "texto_es": "El contrato de trabajo puede suspenderse temporalmente sin responsabilidad para las partes (Art. 65 CT). Causas de suspensión individual: enfermedad del trabajador, licencia otorgada por el patrono, detención o arresto del trabajador. Durante la suspensión el trabajador puede no recibir salario, salvo que la ley o el contrato dispongan lo contrario.",
        "texto_kiche": "Ri ajq'a'nchi' taq tzij ri chak yatob'ën chi tz'apix pa jun rato maj ucholaj pa ri ajchakib' (Art. 65 CT). Rub'eyal chi tz'apix: yawalib'al ri ajchak, licencia xuquje' b'inchixik ri ajchak. Pa ri ratz'aq'at ri tz'apib'al, ri ajchak man k'o ta pwaq, kej ri ley o ri chak nub'ij."
    }
]


In [3]:
import re
import faiss
import numpy as np
import subprocess
from sentence_transformers import SentenceTransformer
from IPython.display import Audio, display

# =========================
# 1) Base de conocimiento: reutilizamos la lista "casos" de la celda anterior
# =========================

# =========================
# 2) Modelo multilingüe
# =========================
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

# =========================
# 3) Preparar índice FAISS
# =========================
def preparar_indice(casos):
    # Indexar título + texto español para mejor recuperación
    textos_indexados = [
        f"{c['titulo']}. {c['texto_es']}" for c in casos
    ]
    emb = modelo.encode(
        textos_indexados,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(emb)
    return index, textos_indexados

index, textos_indexados = preparar_indice(casos)
print(f"Índice listo: {len(casos)} temas cargados del Código de Trabajo de Guatemala.")

# =========================
# 4) Búsqueda semántica
# =========================
def buscar(query, index, casos, k=3):
    q = modelo.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(q, k)

    resultados = []
    for i, sc in zip(ids[0], scores[0]):
        resultados.append({
            "score": float(sc),
            "titulo": casos[i]["titulo"],
            "texto_es": casos[i]["texto_es"],
            "texto_kiche": casos[i]["texto_kiche"]
        })
    return resultados

# =========================
# 5) Audio K'iche' con espeak-ng
# =========================
def generar_audio_kiche(texto, salida="respuesta_kiche.wav"):
    subprocess.run(
        ["espeak-ng", "-v", "quc", "-w", salida, texto],
        check=True
    )
    return salida

# =========================
# 6) Función principal de respuesta
# =========================
def responder(consulta, index, casos, umbral=0.15, mostrar_top=3):
    resultados = buscar(consulta, index, casos, k=mostrar_top)

    print("Top resultados encontrados:")
    for r in resultados:
        print(f"  - {r['score']:.4f} | {r['titulo']}")

    mejor = resultados[0]

    if mejor["score"] < umbral:
        return {
            "encontrado": False,
            "score": mejor["score"],
            "titulo": None,
            "respuesta_es": "No encontré información específica sobre tu consulta. Te recomendamos acudir a la Inspección General de Trabajo (MINTRAB) o a un abogado laboralista.",
            "respuesta_kiche": "Man xinta ta jun k'utunem utz pa ri awetamab'al. Kojkäj pa Inspección General de Trabajo o ri ajcholb'äl tzij.",
            "audio": None
        }

    audio = generar_audio_kiche(mejor["texto_kiche"])

    return {
        "encontrado": True,
        "score": mejor["score"],
        "titulo": mejor["titulo"],
        "respuesta_es": mejor["texto_es"],
        "respuesta_kiche": mejor["texto_kiche"],
        "audio": audio
    }

# =========================
# 7) Prueba rápida
# =========================
consulta_prueba = "¿cuánto me pagan si me despiden sin causa?"
resultado = responder(consulta_prueba, index, casos, umbral=0.15)

print("\n--- RESULTADO ---")
print("Encontrado:", resultado["encontrado"])
print("Score:     ", resultado["score"])
print("Título:    ", resultado["titulo"])
print("ES:        ", resultado["respuesta_es"])
print("K'iche':   ", resultado["respuesta_kiche"])

if resultado["audio"]:
    display(Audio(resultado["audio"]))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Índice listo: 15 temas cargados del Código de Trabajo de Guatemala.
Top resultados encontrados:
  - 0.6882 | Despido injustificado
  - 0.5727 | Renuncia voluntaria
  - 0.5484 | Aguinaldo

--- RESULTADO ---
Encontrado: True
Score:      0.6882306933403015
Título:     Despido injustificado
ES:         Si un trabajador es despedido sin causa justificada (Art. 78 CT), tiene derecho a indemnización equivalente a un mes de salario por cada año de servicio continuo, más el pago de vacaciones, aguinaldo y bono 14 proporcionales. La denuncia puede presentarse en la Inspección General de Trabajo (MINTRAB).
K'iche':    We jun ajchak xel b'anik maj k'utunem (Art. 78 CT), k'o ri rajawarem richin k'amowik pwaq: jun ik' pwaq pa jujun junab' xchak pa ri chak. Xuquje' k'o chi tojik ri ejqanem, aguinaldo xuquje' bono 14. Kojkäj pa Inspección General de Trabajo.


In [4]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇
added 22 packages in 2s
⠇
⠇3 packages are looking for funding
⠇  run `npm fund` for details
⠇

In [5]:
html = r"""
<!DOCTYPE html>
<html lang="es">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>Chatbot de Voz Legal — Guatemala</title>
  <link rel="preconnect" href="https://fonts.googleapis.com" />
  <link href="https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600&family=DM+Serif+Display&display=swap" rel="stylesheet" />
  <style>
    /* ── Reset & Base ── */
    *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

    :root {
      --teal-50:  #E1F5EE;
      --teal-100: #9FE1CB;
      --teal-400: #1D9E75;
      --teal-600: #0F6E56;
      --teal-800: #085041;
      --gray-50:  #F7F6F2;
      --gray-100: #ECEAE3;
      --gray-200: #D3D1C7;
      --gray-400: #888780;
      --gray-600: #5F5E5A;
      --gray-900: #1E1E1C;
      --amber-400: #EF9F27;
      --blue-400:  #378ADD;
      --font-main: 'DM Sans', sans-serif;
      --font-display: 'DM Serif Display', serif;
      --radius-sm: 8px;
      --radius-md: 12px;
      --radius-lg: 18px;
    }

    body {
      font-family: var(--font-main);
      background: var(--gray-50);
      color: var(--gray-900);
      min-height: 100vh;
      display: flex;
      flex-direction: column;
      align-items: center;
      padding: 2rem 1rem 4rem;
    }

    /* ── Page Header ── */
    .page-header {
      width: 100%;
      max-width: 680px;
      margin-bottom: 1.5rem;
      text-align: center;
    }

    .page-header .badge {
      display: inline-block;
      font-size: 11px;
      letter-spacing: 0.08em;
      text-transform: uppercase;
      color: var(--teal-600);
      background: var(--teal-50);
      border: 1px solid var(--teal-100);
      padding: 4px 12px;
      border-radius: 20px;
      margin-bottom: 12px;
      font-weight: 500;
    }

    .page-header h1 {
      font-family: var(--font-display);
      font-size: clamp(22px, 5vw, 30px);
      font-weight: 400;
      color: var(--gray-900);
      line-height: 1.2;
      margin-bottom: 8px;
    }

    .page-header p {
      font-size: 14px;
      color: var(--gray-600);
      line-height: 1.6;
    }

    /* ── Main Card ── */
    .card {
      width: 100%;
      max-width: 680px;
      background: #fff;
      border: 1px solid var(--gray-200);
      border-radius: var(--radius-lg);
      overflow: hidden;
      box-shadow: 0 2px 12px rgba(0,0,0,0.06);
    }

    /* ── Card Top Bar ── */
    .card-topbar {
      display: flex;
      align-items: center;
      justify-content: space-between;
      padding: 14px 20px;
      border-bottom: 1px solid var(--gray-100);
      background: #fff;
    }

    .topbar-left {
      display: flex;
      align-items: center;
      gap: 10px;
    }

    .topbar-icon {
      width: 36px;
      height: 36px;
      border-radius: 50%;
      background: var(--teal-50);
      border: 1px solid var(--teal-100);
      display: flex;
      align-items: center;
      justify-content: center;
    }

    .topbar-icon svg { width: 18px; height: 18px; stroke: var(--teal-600); fill: none; stroke-width: 1.8; stroke-linecap: round; stroke-linejoin: round; }

    .topbar-title { font-size: 14px; font-weight: 600; color: var(--gray-900); }
    .topbar-sub   { font-size: 11px; color: var(--gray-400); }

    .online-dot {
      width: 8px; height: 8px;
      border-radius: 50%;
      background: var(--teal-400);
      animation: pulse-dot 2s infinite;
    }

    @keyframes pulse-dot {
      0%, 100% { opacity: 1; }
      50%       { opacity: 0.4; }
    }

    /* ── Language Selector ── */
    .lang-bar {
      display: flex;
      gap: 6px;
      align-items: center;
      padding: 10px 20px;
      border-bottom: 1px solid var(--gray-100);
      background: var(--gray-50);
    }

    .lang-label { font-size: 11px; color: var(--gray-400); margin-right: 4px; text-transform: uppercase; letter-spacing: 0.05em; }

    .lang-btn {
      font-size: 12px;
      font-family: var(--font-main);
      padding: 5px 14px;
      border-radius: 20px;
      border: 1px solid var(--gray-200);
      background: #fff;
      color: var(--gray-600);
      cursor: pointer;
      transition: all 0.15s;
      font-weight: 400;
    }

    .lang-btn:hover { border-color: var(--teal-100); color: var(--teal-600); }

    .lang-btn.active {
      background: var(--teal-50);
      border-color: var(--teal-400);
      color: var(--teal-800);
      font-weight: 500;
    }

    /* ── Chat Window ── */
    .chat-window {
      padding: 20px;
      min-height: 300px;
      max-height: 400px;
      overflow-y: auto;
      display: flex;
      flex-direction: column;
      gap: 16px;
      background: #fff;
      scroll-behavior: smooth;
    }

    .chat-window::-webkit-scrollbar { width: 4px; }
    .chat-window::-webkit-scrollbar-thumb { background: var(--gray-200); border-radius: 4px; }

    /* ── Messages ── */
    .msg { display: flex; gap: 10px; align-items: flex-start; animation: msg-in 0.22s ease; }
    .msg.user { flex-direction: row-reverse; }

    @keyframes msg-in {
      from { opacity: 0; transform: translateY(6px); }
      to   { opacity: 1; transform: translateY(0); }
    }

    .avatar {
      width: 32px; height: 32px;
      border-radius: 50%;
      display: flex; align-items: center; justify-content: center;
      flex-shrink: 0;
      font-size: 12px; font-weight: 600;
    }

    .avatar.bot  { background: var(--teal-50); border: 1px solid var(--teal-100); color: var(--teal-800); }
    .avatar.user { background: var(--gray-100); border: 1px solid var(--gray-200); color: var(--gray-600); }

    .avatar svg { width: 15px; height: 15px; stroke: currentColor; fill: none; stroke-width: 1.8; stroke-linecap: round; stroke-linejoin: round; }

    .bubble {
      max-width: 75%;
      padding: 11px 15px;
      border-radius: var(--radius-md);
      font-size: 13.5px;
      line-height: 1.6;
      border: 1px solid var(--gray-100);
    }

    .bubble.bot  { background: #fff; border-radius: 4px 14px 14px 14px; }
    .bubble.user { background: var(--teal-50); border-color: var(--teal-100); color: var(--teal-800); border-radius: 14px 4px 14px 14px; text-align: right; }

    .lang-tag {
      display: inline-block;
      font-size: 10px; font-weight: 600;
      text-transform: uppercase; letter-spacing: 0.06em;
      padding: 2px 8px;
      border-radius: 20px;
      background: var(--teal-50);
      border: 1px solid var(--teal-100);
      color: var(--teal-600);
      margin-bottom: 6px;
    }

    .translation {
      margin-top: 9px;
      padding-top: 9px;
      border-top: 1px solid var(--gray-100);
      font-size: 12px;
      color: var(--gray-600);
      font-style: italic;
      line-height: 1.5;
    }

    .translation strong { display: block; font-size: 10px; font-style: normal; text-transform: uppercase; letter-spacing: 0.05em; color: var(--teal-400); margin-bottom: 3px; }

    /* ── Typing indicator ── */
    .typing { display: flex; gap: 4px; align-items: center; padding: 4px 0; }
    .typing span {
      width: 6px; height: 6px; border-radius: 50%;
      background: var(--gray-400);
      animation: bounce 1.2s infinite;
    }
    .typing span:nth-child(2) { animation-delay: 0.15s; }
    .typing span:nth-child(3) { animation-delay: 0.3s; }
    @keyframes bounce {
      0%, 80%, 100% { transform: translateY(0); }
      40%            { transform: translateY(-5px); }
    }



    /* ── Status Bar ── */
    .status-bar {
      display: flex;
      align-items: center;
      gap: 10px;
      padding: 10px 20px;
      border-top: 1px solid var(--gray-100);
      border-bottom: 1px solid var(--gray-100);
      background: #fff;
      font-size: 12px;
      color: var(--gray-400);
      min-height: 38px;
    }

    .status-dot {
      width: 7px; height: 7px;
      border-radius: 50%;
      background: var(--gray-200);
      flex-shrink: 0;
      transition: background 0.3s;
    }

    .status-dot.listening  { background: var(--teal-400); animation: pulse-dot 1s infinite; }
    .status-dot.processing { background: var(--amber-400); animation: pulse-dot 0.7s infinite; }
    .status-dot.speaking   { background: var(--blue-400);  animation: pulse-dot 1.1s infinite; }

    /* ── Controls ── */
    .controls {
      display: flex;
      gap: 10px;
      align-items: center;
      padding: 14px 20px;
      background: #fff;
    }

    .mic-btn {
      width: 54px; height: 54px;
      border-radius: 50%;
      border: 1.5px solid var(--gray-200);
      background: #fff;
      cursor: pointer;
      display: flex; align-items: center; justify-content: center;
      flex-shrink: 0;
      position: relative;
      transition: all 0.2s;
    }

    .mic-btn svg { width: 22px; height: 22px; stroke: var(--gray-600); fill: none; stroke-width: 1.8; stroke-linecap: round; stroke-linejoin: round; transition: stroke 0.2s; }

    .mic-btn:hover { border-color: var(--teal-400); }
    .mic-btn:hover svg { stroke: var(--teal-400); }

    .mic-btn.active {
      border: 2px solid var(--teal-400);
      background: var(--teal-50);
    }

    .mic-btn.active svg { stroke: var(--teal-600); }

    .mic-btn.active::after {
      content: '';
      position: absolute;
      inset: -8px;
      border-radius: 50%;
      border: 2px solid var(--teal-400);
      opacity: 0;
      animation: ripple 1.4s ease-out infinite;
    }

    @keyframes ripple {
      0%   { transform: scale(1); opacity: 0.5; }
      100% { transform: scale(1.6); opacity: 0; }
    }

    .input-row {
      display: flex;
      flex: 1;
      gap: 8px;
    }

    .text-input {
      flex: 1;
      height: 42px;
      font-size: 13.5px;
      font-family: var(--font-main);
      padding: 0 14px;
      border-radius: var(--radius-md);
      border: 1px solid var(--gray-200);
      background: var(--gray-50);
      color: var(--gray-900);
      outline: none;
      transition: border 0.15s;
    }

    .text-input::placeholder { color: var(--gray-400); }
    .text-input:focus { border-color: var(--teal-400); background: #fff; }

    .send-btn {
      height: 42px;
      width: 42px;
      border-radius: var(--radius-md);
      border: 1px solid var(--gray-200);
      background: #fff;
      cursor: pointer;
      display: flex; align-items: center; justify-content: center;
      transition: all 0.15s;
      flex-shrink: 0;
    }

    .send-btn svg { width: 18px; height: 18px; stroke: var(--gray-600); fill: none; stroke-width: 2; stroke-linecap: round; stroke-linejoin: round; }
    .send-btn:hover { background: var(--teal-50); border-color: var(--teal-400); }
    .send-btn:hover svg { stroke: var(--teal-600); }

    /* ── Quick Topics ── */
    .quick-topics {
      padding: 14px 20px 20px;
      background: var(--gray-50);
      border-top: 1px solid var(--gray-100);
    }

    .qt-label {
      font-size: 10px;
      text-transform: uppercase;
      letter-spacing: 0.07em;
      color: var(--gray-400);
      margin-bottom: 10px;
    }

    .chips { display: flex; flex-wrap: wrap; gap: 7px; }

    .chip {
      font-size: 12px;
      font-family: var(--font-main);
      padding: 6px 13px;
      border-radius: 20px;
      border: 1px solid var(--gray-200);
      background: #fff;
      color: var(--gray-600);
      cursor: pointer;
      transition: all 0.15s;
    }

    .chip:hover {
      background: var(--teal-50);
      border-color: var(--teal-100);
      color: var(--teal-800);
    }

    /* ── Footer ── */
    .page-footer {
      max-width: 680px;
      width: 100%;
      margin-top: 16px;
      font-size: 11px;
      color: var(--gray-400);
      text-align: center;
      line-height: 1.7;
    }

    /* ── Botones de Audio ── */
    .audio-btns {
      display: flex;
      gap: 8px;
      margin-top: 12px;
      flex-wrap: wrap;
    }

    .audio-btn {
      font-size: 12px;
      font-family: var(--font-main);
      padding: 6px 14px;
      border-radius: 20px;
      border: 1px solid var(--gray-200);
      background: #fff;
      color: var(--gray-600);
      cursor: pointer;
      transition: all 0.15s;
      display: flex;
      align-items: center;
      gap: 5px;
    }

    .audio-btn:hover {
      background: var(--teal-50);
      border-color: var(--teal-400);
      color: var(--teal-800);
    }

    .audio-btn.kiche {
      border-color: var(--amber-400);
      color: #b45309;
    }

    .audio-btn.kiche:hover {
      background: #fffbeb;
      border-color: var(--amber-400);
      color: #92400e;
    }

    /* ── Responsive ── */
    @media (max-width: 500px) {
      body { padding: 1rem 0.5rem 3rem; }
      .bubble { max-width: 88%; }
    }
  </style>
</head>
<body>

  <!-- Page Header -->
  <header class="page-header">
    <div class="badge">Prototipo · Trabajo de Graduación USAC 2025</div>
    <h1>Orientación Legal Laboral</h1>
    <p>Asistente de voz multilingüe · Español &amp; K'iche'</p>
  </header>

  <!-- Main Card -->
  <main class="card" role="main">

    <!-- Top Bar -->
    <div class="card-topbar">
      <div class="topbar-left">
        <div class="topbar-icon">
          <!-- Scale icon -->
          <svg viewBox="0 0 24 24"><path d="M12 3v18M6 7l-3 7a3 3 0 006 0L6 7zM18 7l-3 7a3 3 0 006 0L18 7zM6 7h12M8 19h8"/></svg>
        </div>
        <div>
          <div class="topbar-title">Asistente Legal Laboral</div>
          <div class="topbar-sub">Español · K'iche'</div>
        </div>
      </div>
      <div class="online-dot" title="Sistema activo"></div>
    </div>

    <!-- Language Selector -->
    <div class="lang-bar" role="toolbar" aria-label="Selector de idioma">
      <span class="lang-label">Idioma</span>
      <button class="lang-btn active" id="btn-es" onclick="setLang('es')" aria-pressed="true">Español</button>
      <button class="lang-btn" id="btn-ki" onclick="setLang('ki')" aria-pressed="false">K'iche'</button>
      <button class="lang-btn" id="btn-bi" onclick="setLang('bi')" aria-pressed="false">Bilingüe</button>
    </div>

    <!-- Chat Window -->
    <section class="chat-window" id="chat-window" aria-live="polite" aria-label="Conversación">
      <!-- Mensaje inicial del bot -->
      <div class="msg bot">
        <div class="avatar bot">
          <svg viewBox="0 0 24 24"><path d="M12 3v18M6 7l-3 7a3 3 0 006 0L6 7zM18 7l-3 7a3 3 0 006 0L18 7zM6 7h12M8 19h8"/></svg>
        </div>
        <div class="bubble bot" id="welcome-bubble">
          <div class="lang-tag">Español</div>
          <p>Bienvenido/a. Soy tu asistente de orientación legal laboral. Puedes preguntarme sobre tus derechos según el <strong>Código de Trabajo de Guatemala</strong>: salarios, despidos, vacaciones, contratos y más.</p>
          <div class="translation">
            <strong>K'iche'</strong>
            Ütz awilabäl. Yin ajwachib'äl pa ri ajq'a'nchi' taq tzij. Kojkäj chech ri awetamab'al chirej ri awokisaxik pa Código de Trabajo Guatemala.
          </div>
        </div>
      </div>
    </section>

    <!-- Status Bar -->
    <div class="status-bar">
      <div class="status-dot" id="status-dot"></div>
      <span id="status-text">Presiona el micrófono o escribe tu consulta</span>
    </div>

    <!-- Controls -->
    <div class="controls">
      <button class="mic-btn" id="mic-btn" onclick="toggleMic()" aria-label="Activar o desactivar micrófono" title="Mantén pulsado para hablar">
        <!-- Mic icon (default) -->
        <svg id="mic-icon" viewBox="0 0 24 24"><path d="M12 2a3 3 0 013 3v7a3 3 0 01-6 0V5a3 3 0 013-3z"/><path d="M19 10v2a7 7 0 01-14 0v-2M12 19v3M8 22h8"/></svg>
      </button>
      <div class="input-row">
        <input
          type="text"
          class="text-input"
          id="text-input"
          placeholder="Escribe o usa el micrófono para consultar…"
          aria-label="Escribe tu consulta"
          onkeydown="if(event.key==='Enter') sendText()"
        />
        <button class="send-btn" onclick="sendText()" aria-label="Enviar consulta">
          <svg viewBox="0 0 24 24"><line x1="22" y1="2" x2="11" y2="13"/><polygon points="22 2 15 22 11 13 2 9 22 2"/></svg>
        </button>
      </div>
    </div>

    <!-- Quick Topics -->
    <div class="quick-topics">
      <p class="qt-label">Temas frecuentes</p>
      <div class="chips">
        <button class="chip" onclick="askTopic('salario mínimo')">Salario mínimo</button>
        <button class="chip" onclick="askTopic('despido injustificado')">Despido injustificado</button>
        <button class="chip" onclick="askTopic('vacaciones')">Vacaciones</button>
        <button class="chip" onclick="askTopic('jornada de trabajo')">Jornada laboral</button>
        <button class="chip" onclick="askTopic('contrato de trabajo')">Contrato de trabajo</button>
        <button class="chip" onclick="askTopic('aguinaldo')">Aguinaldo</button>
        <button class="chip" onclick="askTopic('bono 14')">Bono 14</button>
        <button class="chip" onclick="askTopic('maternidad')">Maternidad / Paternidad</button>
        <button class="chip" onclick="askTopic('seguro social')">Seguro social IGSS</button>
        <button class="chip" onclick="askTopic('discriminación')">Discriminación laboral</button>
        <button class="chip" onclick="askTopic('días de asueto')">Días de asueto</button>
        <button class="chip" onclick="askTopic('trabajo menores')">Trabajo de menores</button>
        <button class="chip" onclick="askTopic('renuncia')">Renuncia voluntaria</button>
      </div>
    </div>

  </main>

  <!-- Page Footer -->
  <footer class="page-footer">
    Prototipo académico · Licda. Maria Fernanda Gadea Letona · USAC 2025<br>
    Este sistema es orientativo y no reemplaza asesoría legal profesional.
  </footer>


  <script>
    /* ══════════════════════════════════════════════
       BASE DE CONOCIMIENTO LEGAL
       Fuente: Código de Trabajo de Guatemala
       Respuestas en Español + K'iche'
    ══════════════════════════════════════════════ */
    var KB = {
      'salario mínimo': {
        es: 'El empleador no puede pagar menos del salario mínimo vigente (Art. 103 CT). Para 2025: actividades no agrícolas Q11.74/hora y agrícolas Q11.52/hora. El salario mínimo se fija anualmente por el Organismo Ejecutivo. El incumplimiento es sancionable por el MINTRAB.',
        ki: "Ri ajpatron man utz taj nutoj jub'a' pwaq chuwäch ri pwaq ri k'o chi tojik pa ruwi' ri ley (Art. 103 CT). 2025: Q11.74 pa hora ri chak chi najt pa poqonal. Ri tz'aq'at uwetamab'al k'iyin pa ronojel junab'. Ri ajpatron ri man nutoj ta ri tz'aq'at uwetamab'al, k'o chi nuk'is ri multa."
      },
      'despido injustificado': {
        es: 'Si fuiste despedido sin causa justificada (Art. 78 CT), tienes derecho a: indemnización de 1 mes de salario por cada año trabajado, más vacaciones, aguinaldo y bono 14 proporcionales. Puedes presentar tu denuncia en la Inspección General de Trabajo (MINTRAB). El plazo para reclamar es de 2 años.',
        ki: "We xetz'apij awib' roma majun ucholaj (Art. 78 CT), xatilitaj pa: indemnización jun ik' pwaq pa jujun junab', xuquje' ejqanem, aguinaldo, bono 14. Kojkäj pa Inspección General de Trabajo. Kab' juna' (2 juna') chi kojkäj."
      },
      'vacaciones': {
        es: 'Tienes derecho a 15 días hábiles de vacaciones pagadas por cada año de trabajo continuo (Art. 130 CT). Deben tomarse dentro de los 60 días siguientes al año cumplido. Si cesás antes del año, recibís compensación proporcional. No pueden compensarse en dinero mientras estés trabajando.',
        ki: "Xatilitaj pa chib'ojlajuj (15) q'ij e okisaxik chi ejqanem tojonik pa junam jun juna' e chakïk (Art. 130 CT). K'o chi k'amowik pa wuqajinik (60) q'ij chik rij ri juna'. We kel pa ri chak nab'e juna', k'o rajawarem richin nik'aj ejqanem. Man nitz'aqatex ta rech pwaq chi e b'antaj."
      },
      'jornada de trabajo': {
        es: 'La jornada ordinaria diurna es de 8 horas diarias y 44 horas semanales (Art. 116 CT). La nocturna es de 6 horas diarias y 36 semanales. La mixta es de 7 horas diarias y 42 semanales. No pueden exceder 12 horas diarias en total. Las horas extras se pagan con recargo mínimo del 50%.',
        ki: "Ri chakïk pa q'ij: waqxaqib' (8) hora pa jun q'ij, kuk' kawinaq kab' (44) hora pa junab'aq'a (Art. 116 CT). Pa aq'ab': waqib' (6) hora pa jun q'ij, kuk' wuqajinik waqib' (36). Man k'o chi nuk'is ta pa lajuj kab' (12) hora pa jun q'ij. Ri taq hora k'utun rajawaxik nik'aj chik ri pwaq."
      },
      'contrato de trabajo': {
        es: 'El contrato puede ser verbal o escrito (Art. 26 CT), pero el escrito ofrece mayor protección. Debe incluir: nombre de las partes, descripción del puesto, salario, lugar y duración. Existe presunción de relación laboral cuando hay subordinación. El período de prueba máximo es de 2 meses.',
        ki: "Ri ajq'a'nchi' taq tzij yatob'ën pa tzib'b'äl o pa tzij (Art. 26 CT). Ri pa tzib'b'äl k'o ri nima'q rajawaxik. K'o chi nuk'ulb'esaj: ri b'i'aj, ri tz'aq'at uwetamab'al, ri qas taq tzij. Ri k'atz'inem tz'aq'at uwetamab'al kab' ik' (2 ik')."
      },
      'aguinaldo': {
        es: 'El aguinaldo equivale a un salario mensual íntegro y debe pagarse del 1 al 15 de diciembre de cada año (Decreto 76-78). Si laboraste menos de un año, recibes la parte proporcional. No puede sustituirse ni deducirse de otros pagos.',
        ki: "Ri aguinaldo rajawaxik jujun mes tz'aq'at uwetamab'al, nub'an pa ri q'ij nik'aj lajuj Diciembre pa jujun juna' (Decreto 76-78). We mayub' juna' xab'an chakïk, rajawaxik nik'aj aguinaldo. Man nitz'aqatex ta rech jun chik pwaq."
      },
      'bono 14': {
        es: 'El Bono 14 equivale a un salario mensual y debe pagarse en la primera quincena de julio (Decreto 42-92). Si trabajaste menos de un año, corresponde la parte proporcional. Se calcula con el salario promedio de los últimos 12 meses. No puede sustituirse ni deducirse.',
        ki: "Ri Bono 14 rajawaxik jujun mes tz'aq'at uwetamab'al, nub'an pa ri nik'aj b'elejeb' winaq Julio pa jujun juna' (Decreto 42-92). We mayub' juna' xab'an chakïk, rajawaxik nik'aj bono. Nub'an pa ruwi' ri pwaq promedio ri lajuj kab' ik' (12 ik')."
      },
      'maternidad': {
        es: 'La trabajadora embarazada tiene derecho a 84 días de descanso remunerado: 30 días antes y 54 días después del parto (Art. 152 CT). El patrono no puede despedirla durante el embarazo ni dentro de los 10 meses posteriores al parto. El padre tiene derecho a 2 días de licencia pagada.',
        ki: "Ri ixoq ajchak iyom xatilitaj pa wuqub' winaq (84 q'ij) e oq'omaxik chi iyomaxik: wuqajinik (30) q'ij nab'e ri alaxik, xuquje' lajuj wuqajinik (54) q'ij chik rij (Art. 152 CT). Man xtz'apij ta ri ixoq iyom pa ri chak. Ri achi k'o rajawarem richin kab' (2) q'ij ejqanem."
      },
      'seguro social': {
        es: 'El patrono debe afiliar a sus trabajadores al IGSS y realizar los aportes correspondientes. El IGSS cubre: enfermedad, maternidad, invalidez, vejez y accidentes. El trabajador también aporta un porcentaje de su salario. El incumplimiento patronal es sancionable.',
        ki: "Ri ajpatron k'o chi rutz'aqatisaxik ri ajchak pa IGSS xuquje' k'o chi rutojik ri k'iyojil. Ri IGSS nuchap: yawalib'al, iyomaxik, pixab'anel, xuquje' tzaq'axik pa ri chak. Ri ajchak xuquje' k'o nutoj ri rajilab'al. Ri ajpatron ri man nub'an ta, k'o chi nuk'is multa."
      },
      'discriminación': {
        es: 'Se prohíbe la discriminación por motivo de raza, religión, credos políticos, sexo, situación económica o etnia (Art. 14 BIS CT). El patrono que discrimine en la contratación o durante la relación laboral puede ser sancionado con el doble del salario dejado de percibir.',
        ki: "Man utz ta ri k'exb'al ucholaj ajchak pa ruwi' ri amaq', ri keroj, ri achi o ixoq, ri tz'aq'at uwetamab'al (Art. 14 BIS CT). Ri ajpatron ri nuk'ex cholaj k'o chi nuk'is multa ruk' kab' (2) chik ri pwaq."
      },
      'días de asueto': {
        es: 'Los trabajadores particulares tienen derecho a días de asueto pagados (Art. 127 CT): 1 enero, Semana Santa (jueves-sábado), 1 mayo, 30 junio, 15 septiembre, 20 octubre, 1 noviembre, 24 diciembre (medio día), 25 diciembre, 31 diciembre (medio día) y festividad local.',
        ki: "Ri taq ajchak k'o rajawarem richin taq q'ij asueto ruk' tojib'al pwaq (Art. 127 CT): ri nab'e q'ij Enero, Semana Santa, ri nab'e q'ij Mayo, ri wuqajinik q'ij Junio, ri lajuj wuqub' q'ij Septiembre, ri winaq q'ij Octubre, ri nab'e q'ij Noviembre, xuquje' ri q'ij festividad."
      },
      'trabajo menores': {
        es: 'Se prohíbe el trabajo de menores de 14 años (Art. 148 CT). Para mayores de 14 años la jornada se reduce en 1 hora diaria. Se prohíbe el trabajo nocturno y extraordinario para menores. El MINTRAB puede autorizar excepciones. Los menores deben cumplir con la obligatoriedad escolar.',
        ki: "Man k'o ta chi chak ri achike ajchak man k'ulmataj ta lajuj kaib' (14) junab' (Art. 148 CT). Ri más kab'lajuj junab', ri q'ijil chak nuk'is jun hora pa jun q'ij. Man k'o ta chi chak pa aq'ab'. K'o chi nab'an chak pa ri tijob'al."
      },
      'renuncia': {
        es: 'Si el trabajador renuncia voluntariamente, no hay indemnización. Pero el patrono debe pagar: vacaciones no gozadas, proporciones de aguinaldo y bono 14, y cualquier salario adeudado. Si el patrono incumple el contrato, el trabajador puede retirarse con derecho a indemnización (Art. 79 CT).',
        ki: "We ri ajchak kel ruk' ri chak pa ruma ri utzil, man k'o ta indemnización. Pero k'o chi tojik ri rajilab'al k'iyojil k'o chik: ejqanem, nik'aj aguinaldo xuquje' bono 14. We ri ajpatron man nuk'umaj ta ri chak, ri ajchak k'o rajawarem richin indemnización (Art. 79 CT)."
      }
    };

    /* ══════════════════════════════════════════════
       ESTADO DE LA APP
    };

    /* ══════════════════════════════════════════════
       ESTADO DE LA APP
    ══════════════════════════════════════════════ */
    var currentLang = 'es';
    var micActive   = false;
    var pipeTimer   = null;

    /* ══════════════════════════════════════════════
       SELECTOR DE IDIOMA
    ══════════════════════════════════════════════ */
    function setLang(lang) {
      currentLang = lang;
      ['es','ki','bi'].forEach(function(l) {
        var btn = document.getElementById('btn-' + l);
        btn.classList.toggle('active', l === lang);
        btn.setAttribute('aria-pressed', l === lang ? 'true' : 'false');
      });
      // Actualizar burbuja de bienvenida según idioma
      var wb = document.getElementById('welcome-bubble');
      if (!wb) return;
      var tag = wb.querySelector('.lang-tag');
      var tr  = wb.querySelector('.translation');
      if (lang === 'es') {
        tag.textContent = 'Español';
        if (tr) tr.style.display = 'block';
      } else if (lang === 'ki') {
        tag.textContent = "K'iche'";
        if (tr) tr.style.display = 'none';
      } else {
        tag.textContent = 'Español / K\'iche\'';
        if (tr) tr.style.display = 'block';
      }
    }

    /* ══════════════════════════════════════════════
       MICRÓFONO REAL — MediaRecorder + Whisper
    ══════════════════════════════════════════════ */
    var mediaRecorder = null;
    var audioChunks   = [];

    async function toggleMic() {
      var btn  = document.getElementById('mic-btn');
      var icon = document.getElementById('mic-icon');

      // ── Si ya está grabando: DETENER ──
      if (micActive) {
        micActive = false;
        btn.classList.remove('active');
        icon.innerHTML = '<path d="M12 2a3 3 0 013 3v7a3 3 0 01-6 0V5a3 3 0 013-3z"/><path d="M19 10v2a7 7 0 01-14 0v-2M12 19v3M8 22h8"/>';
        if (mediaRecorder && mediaRecorder.state !== 'inactive') {
          mediaRecorder.stop();   // dispara ondataavailable + onstop
        }
        return;
      }

      // ── Solicitar acceso al micrófono ──
      let stream;
      try {
        stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      } catch (err) {
        setStatus('idle', '⚠ No se pudo acceder al micrófono: ' + err.message);
        return;
      }

      // ── Iniciar grabación ──
      micActive    = true;
      audioChunks  = [];
      mediaRecorder = new MediaRecorder(stream);

      btn.classList.add('active');
      icon.innerHTML = '<path d="M1 1l22 22M9 9v3a3 3 0 005.12 2.12M15 9.34V4a3 3 0 00-5.94-.6M17 16.95A7 7 0 015 12v-2m14 0v2a7 7 0 01-.11 1.23M12 19v3M8 22h8"/>';
      setStatus('listening', '🔴 Grabando… presiona de nuevo para enviar');

      mediaRecorder.ondataavailable = function(e) {
        if (e.data.size > 0) audioChunks.push(e.data);
      };

      mediaRecorder.onstop = async function() {
        // Detener tracks del micrófono
        stream.getTracks().forEach(t => t.stop());
        setStatus('processing', 'Transcribiendo con Whisper…');

        // Armar el Blob de audio y enviarlo al backend
        var blob     = new Blob(audioChunks, { type: 'audio/webm' });
        var formData = new FormData();
        formData.append('audio', blob, 'grabacion.webm');

        try {
          var resp = await fetch('/transcribir', { method: 'POST', body: formData });
          var data = await resp.json();
          var texto = data.texto || '';
          var idioma = data.idioma || '';

          if (!texto) {
            setStatus('idle', 'No se detectó voz. Intenta de nuevo.');
            return;
          }

          // Mostrar transcripción en el chat y procesarla
          setStatus('processing', 'Buscando respuesta… (idioma detectado: ' + idioma + ')');
          addUserMsg('🎙 ' + texto);

          var r2 = await fetch('/texto', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ texto: texto })
          });
          var respData = await r2.json();
          addBotMsg(respData);
          setStatus('idle', 'Presiona el micrófono o escribe tu consulta');

          // Reproducir respuesta en K'iche'
          await reproducirAudio(texto);

        } catch(err) {
          setStatus('idle', 'Error al procesar audio: ' + err.message);
        }
      };

      mediaRecorder.start();
    }

    /* ══════════════════════════════════════════════
       STATUS BAR
    ══════════════════════════════════════════════ */
    function setStatus(type, text) {
      var dot = document.getElementById('status-dot');
      dot.className = 'status-dot' + (type !== 'idle' ? ' ' + type : '');
      document.getElementById('status-text').textContent = text;
    }

    /* ══════════════════════════════════════════════
       PROCESAMIENTO (interno — sin pipeline visible)
    ══════════════════════════════════════════════ */
    function animatePipeline(onDone) {
      if (pipeTimer) clearTimeout(pipeTimer);
      setStatus('processing', 'Buscando información…');
      pipeTimer = setTimeout(function() {
        setStatus('speaking', 'Preparando respuesta…');
        pipeTimer = setTimeout(function() {
          setStatus('idle', 'Presiona el micrófono o escribe tu consulta');
          if (onDone) onDone();
        }, 800);
      }, 1400);
    }

    /* ══════════════════════════════════════════════
       MENSAJES: USUARIO
    ══════════════════════════════════════════════ */
    function addUserMsg(text) {
      var chat = document.getElementById('chat-window');
      var div  = document.createElement('div');
      div.className = 'msg user';
      div.innerHTML =
        '<div class="avatar user"><svg viewBox="0 0 24 24" style="width:15px;height:15px;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round"><path d="M20 21v-2a4 4 0 00-4-4H8a4 4 0 00-4 4v2M12 11a4 4 0 100-8 4 4 0 000 8z"/></svg></div>' +
        '<div class="bubble user">' + escHtml(text) + '</div>';
      chat.appendChild(div);
      chat.scrollTop = chat.scrollHeight;
    }

    /* ══════════════════════════════════════════════
       MENSAJES: BOT (con typing indicator)
    ══════════════════════════════════════════════ */
    function respondBot(topic) {
      var chat = document.getElementById('chat-window');

      // Typing indicator
      var typing = document.createElement('div');
      typing.className = 'msg bot';
      typing.id = 'typing-indicator';
      typing.innerHTML =
        '<div class="avatar bot"><svg viewBox="0 0 24 24" style="width:15px;height:15px;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round"><path d="M12 3v18M6 7l-3 7a3 3 0 006 0L6 7zM18 7l-3 7a3 3 0 006 0L18 7zM6 7h12M8 19h8"/></svg></div>' +
        '<div class="bubble bot"><div class="typing"><span></span><span></span><span></span></div></div>';
      chat.appendChild(typing);
      chat.scrollTop = chat.scrollHeight;

      animatePipeline(function() {
        // Quitar typing indicator
        var ti = document.getElementById('typing-indicator');
        if (ti) ti.remove();

        var resp = KB[topic];
        if (!resp) {
          resp = {
            es: 'Tu consulta sobre "' + topic + '" es importante. Para información más específica, te recomendamos acudir a la Inspección General de Trabajo o consultar con un abogado laboralista.',
            ki: "Ri awetamab'al rech '" + topic + "' rajawaxik. Kojkäj rech ri Inspección General de Trabajo o ri ajcholb'äl tzij."
          };
        }

        var mainText   = currentLang === 'ki' ? (resp.ki || resp.es) : resp.es;
        var langLabel  = currentLang === 'ki' ? "K'iche'" : 'Español';
        var showTrans  = currentLang === 'bi';

        var transHtml = showTrans && resp.ki
          ? '<div class="translation"><strong>K\'iche\'</strong>' + escHtml(resp.ki) + '</div>'
          : '';

        var div = document.createElement('div');
        div.className = 'msg bot';
        div.innerHTML =
          '<div class="avatar bot"><svg viewBox="0 0 24 24" style="width:15px;height:15px;stroke:currentColor;fill:none;stroke-width:1.8;stroke-linecap:round;stroke-linejoin:round"><path d="M12 3v18M6 7l-3 7a3 3 0 006 0L6 7zM18 7l-3 7a3 3 0 006 0L18 7zM6 7h12M8 19h8"/></svg></div>' +
          '<div class="bubble bot">' +
            '<div class="lang-tag">' + langLabel + '</div>' +
            '<p>' + escHtml(mainText) + '</p>' +
            transHtml +
          '</div>';
        chat.appendChild(div);
        chat.scrollTop = chat.scrollHeight;
      });
    }

    /* ══════════════════════════════════════════════
       TEMA RÁPIDO (chips)
    ══════════════════════════════════════════════ */
    function askTopic(topic) {
      addUserMsg(topic.charAt(0).toUpperCase() + topic.slice(1));
      setTimeout(function() { respondBot(topic); }, 500);
    }

    /* ══════════════════════════════════════════════
       ENVÍO DE TEXTO
    ══════════════════════════════════════════════ */
    async function sendText() {
      var input = document.getElementById('text-input');
      var val   = input.value.trim();
      if (!val) return;
      input.value = '';
      addUserMsg(val);

      // Buscar coincidencia en base de conocimiento
       const response = await fetch(
        "/texto",
        {
            method: "POST",
            headers: {
                "Content-Type": "application/json"
            },
            body: JSON.stringify({
                texto: val
            })
        }
    );

      const data = await response.json();

      console.log(data);
      addBotMsg(data);
      // El usuario elige el idioma con los botones de audio
      /*var lv      = val.toLowerCase();
      var matched = Object.keys(KB).find(function(k) { return lv.indexOf(k) >= 0; });
      setTimeout(function() { respondBot(matched || val); }, 500);*/
    }

    /* ══════════════════════════════════════════════
       UTILIDAD: escapar HTML
    ══════════════════════════════════════════════ */
    function escHtml(str) {
      return String(str)
        .replace(/&/g,  '&amp;')
        .replace(/</g,  '&lt;')
        .replace(/>/g,  '&gt;')
        .replace(/"/g,  '&quot;');
    }

    function addBotMsg(data) {
  var chat = document.getElementById('chat-window');

  // Guardar última respuesta para los botones de audio
  window._ultimaRespuesta = data;

  var div = document.createElement('div');
  div.className = 'msg bot';

  div.innerHTML =
    '<div class="avatar bot">⚖️</div>' +
    '<div class="bubble bot">' +
      '<div class="lang-tag">' + escHtml(data.titulo || '') + '</div>' +
      '<p>' + escHtml(data.respuesta_es) + '</p>' +
      '<div class="translation"><strong>K\'iche\'</strong>' +
        escHtml(data.respuesta_kiche) +
      '</div>' +
      '<div class="audio-btns">' +
        '<button class="audio-btn" onclick="escucharIdioma(\'es\')" title="Escuchar en Español">' +
          '🔊 Escuchar en Español' +
        '</button>' +
        '<button class="audio-btn kiche" onclick="escucharIdioma(\'ki\')" title="Escuchar en K\'iche\'">' +
          '🔊 Escuchar en K\'iche\'' +
        '</button>' +
      '</div>' +
    '</div>';

  chat.appendChild(div);
  chat.scrollTop = chat.scrollHeight;
}

// ── Reproducir audio según idioma seleccionado ──
async function escucharIdioma(lang) {
  var data = window._ultimaRespuesta;
  if (!data) return;

  var endpoint = lang === 'ki' ? '/audio-kiche' : '/audio-es';
  var texto    = lang === 'ki' ? data.respuesta_kiche : data.respuesta_es;

  setStatus('speaking', lang === 'ki' ? "Reproduciendo en K'iche'…" : "Reproduciendo en Español…");

  try {
    var resp = await fetch(endpoint, {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ texto: texto })
    });
    var blob     = await resp.blob();
    var audioURL = URL.createObjectURL(blob);
    var audio    = new Audio(audioURL);
    audio.onended = function() {
      setStatus('idle', 'Presiona el micrófono o escribe tu consulta');
    };
    audio.play();
  } catch(err) {
    setStatus('idle', 'Error al reproducir audio: ' + err.message);
  }
}

async function reproducirAudio(texto) {
  const response = await fetch("/audio", {
    method: "POST",
    headers: {
      "Content-Type": "application/json"
    },
    body: JSON.stringify({
      texto: texto
    })
  });

  const blob = await response.blob();
  const audioURL = URL.createObjectURL(blob);
  const audio = new Audio(audioURL);

  audio.play();
}
  </script>

</body>
</html>
"""

with open("index.html", "w", encoding="utf-8") as f:
    f.write(html)

In [6]:
from fastapi import FastAPI, UploadFile, File
from fastapi.responses import FileResponse, HTMLResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import whisper
import tempfile, os, subprocess

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Cargar Whisper una sola vez
print("Cargando modelo Whisper...")
whisper_model = whisper.load_model("medium")
print("Whisper listo.")

class Consulta(BaseModel):
    texto: str

@app.get("/", response_class=HTMLResponse)
def home():
    with open("index.html", "r", encoding="utf-8") as f:
        return f.read()

# ── Transcripción de voz ──
@app.post("/transcribir")
async def endpoint_transcribir(audio: UploadFile = File(...)):
    suffix = os.path.splitext(audio.filename)[1] or ".webm"
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(await audio.read())
        tmp_path = tmp.name
    try:
        result = whisper_model.transcribe(tmp_path, task="transcribe")
        texto  = result["text"].strip()
        idioma = result.get("language", "desconocido")
    finally:
        os.unlink(tmp_path)
    return {"texto": texto, "idioma": idioma}

# ── Búsqueda de respuesta ──
@app.post("/texto")
def endpoint_texto(data: Consulta):
    resultado = responder(data.texto, index, casos, umbral=0.15)
    return {
        "encontrado":      resultado["encontrado"],
        "score":           resultado["score"],
        "titulo":          resultado["titulo"],
        "respuesta_es":    resultado["respuesta_es"],
        "respuesta_kiche": resultado["respuesta_kiche"]
    }

# ── Audio en K'iche' (espeak-ng) ──
@app.post("/audio-kiche")
def endpoint_audio_kiche(data: Consulta):
    resultado = responder(data.texto, index, casos, umbral=0.15)
    if not resultado["encontrado"]:
        return {"error": "No se encontró respuesta."}
    salida = "respuesta_kiche.wav"
    subprocess.run(
        ["espeak-ng", "-v", "quc", "-w", salida, resultado["respuesta_kiche"]],
        check=True
    )
    return FileResponse(salida, media_type="audio/wav", filename="respuesta_kiche.wav")

# ── Audio en Español (espeak-ng voz es) ──
@app.post("/audio-es")
def endpoint_audio_es(data: Consulta):
    resultado = responder(data.texto, index, casos, umbral=0.15)
    if not resultado["encontrado"]:
        return {"error": "No se encontró respuesta."}
    salida = "respuesta_es.wav"
    subprocess.run(
        ["espeak-ng", "-v", "es", "-s", "140", "-w", salida, resultado["respuesta_es"]],
        check=True
    )
    return FileResponse(salida, media_type="audio/wav", filename="respuesta_es.wav")


Cargando modelo Whisper...


100%|██████████████████████████████████████| 1.42G/1.42G [00:08<00:00, 172MiB/s]


Whisper listo.


In [7]:
import nest_asyncio
import uvicorn
from threading import Thread

nest_asyncio.apply()

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = Thread(target=run_api)
thread.daemon = True
thread.start()

In [8]:
%%bash --bg
lt --port 8000 > localtunnel.log 2>&1

In [10]:
!cat localtunnel.log

your url is: https://large-phones-fly.loca.lt
